0: installs

In [ ]:
!pip install -q transformers accelerate torch torchvision opencv-python pandas tqdm openai
!pip install -q git+https://github.com/huggingface/transformers@v4.49.0-SmolVLM-2
!pip install -q num2words av
!pip install -q scikit-learn scipy matplotlib seaborn
!pip install -q sentence-transformers rouge-score tenacity
!pip install -q networkx ipywidgets umap-learn tabulate

print("0 - All dependencies installed!")


1: Imports + Configuration + Paths

In [ ]:
import os, json, time, logging, asyncio, shutil, zipfile, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import nest_asyncio
nest_asyncio.apply()

from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from collections import defaultdict
from tqdm.auto import tqdm

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, normalized_mutual_info_score,
    adjusted_rand_score, accuracy_score,
    precision_score, recall_score, f1_score
)
from scipy.optimize import linear_sum_assignment
from scipy.stats import bootstrap, wilcoxon
import torch


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
logger = logging.getLogger(__name__)


@dataclass
class ExperimentConfig:

    n_runs: int = 15  # (30, 50, 100, ...) for more robust statistical analysis
    n_clusters: int = 20
    test_mode: bool = False  # set True for test_mode
    test_sample_size: int = 50  # Set value if test_mode True

    smolvlm_path: str = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

    llm_models: Dict[str, str] = field(default_factory=lambda: {
        "gemini": "google/gemini-2.5-flash",
        "claude": "anthropic/claude-3-haiku",
        "llama" : "meta-llama/llama-3.3-70b-instruct"
    })


    drive_workspace: str = '/content/drive/MyDrive/MSRVTT_Workspace'
    local_workspace: str = '/content/local_data'

    random_seeds: List[int] = field(default_factory=lambda: None)

    def __post_init__(self):
        if self.random_seeds is None:
            self.random_seeds = [42 + i for i in range(self.n_runs)]

config = ExperimentConfig()


EMBEDDING_MODELS = {
    "qwen"  : "qwen/qwen3-embedding-8b",
    "openai": "openai/text-embedding-3-large"
}


MSRVTT_CATEGORY_MAP = {
    0: 'music', 1: 'people', 2: 'gaming', 3: 'sports/actions',
    4: 'news/events/politics', 5: 'education', 6: 'tv shows',
    7: 'movie/comedy', 8: 'animation', 9: 'vehicles/autos',
    10: 'howto', 11: 'travel', 12: 'science/technology',
    13: 'animals/pets', 14: 'kids/family', 15: 'documentary',
    16: 'food/drink', 17: 'cooking', 18: 'beauty/fashion',
    19: 'advertisement'
}

# ── Paths ───────────────────────────────────────────────────────────────────
OUTPUTS_DIR      = os.path.join(config.drive_workspace, 'outputs')
LOCAL_WORKSPACE  = config.local_workspace
VIDEOS_DIR       = os.path.join(LOCAL_WORKSPACE, 'videos')
EMBEDDINGS_DIR   = os.path.join(OUTPUTS_DIR, 'embeddings')
BASE_CSV         = os.path.join(OUTPUTS_DIR, 'MSRVTT_base.csv')
FINAL_CSV_PATH   = os.path.join(OUTPUTS_DIR, 'MSRVTT_dados_multillm.csv')
ZIP_ON_DRIVE     = os.path.join(config.drive_workspace, 'MSRVTT_Videos.zip')
JSON_ON_DRIVE    = os.path.join(config.drive_workspace, 'MSR_VTT.json')
GLOBAL_SEED      = 42

for d in [OUTPUTS_DIR, LOCAL_WORKSPACE, EMBEDDINGS_DIR]:
    os.makedirs(d, exist_ok=True)

logger.info("CELL 1 COMPLETE")
logger.info(f"   OUTPUTS_DIR   : {OUTPUTS_DIR}")
logger.info(f"   EMBEDDINGS_DIR: {EMBEDDINGS_DIR}")
logger.info(f"   n_runs        : {config.n_runs}")
logger.info(f"   n_clusters    : {config.n_clusters}")
logger.info(f"   Embeddings    : {list(EMBEDDING_MODELS.keys())}")

print("1 — IMPORTS + CONFIG + PATHS: COMPLETED")



2:  Utility Functions

In [ ]:

class ClusteringUtils:

    @staticmethod
    def hungarian_alignment(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:

        cm = confusion_matrix(y_true, y_pred)
        row_ind, col_ind = linear_sum_assignment(-cm)
        mapping = {col: row for row, col in zip(row_ind, col_ind)}
        return np.array([mapping.get(label, label) for label in y_pred])

    @staticmethod
    def compute_stats_with_ci(values: List[float], alpha: float = 0.05) -> Dict:

        values = np.array(values)
        mean   = float(np.mean(values))
        std    = float(np.std(values, ddof=1))

        def _statistic(x, axis):
            return np.mean(x, axis=axis)

        res = bootstrap(
            (values,),
            _statistic,
            n_resamples=1000,
            confidence_level=1 - alpha,
            method='percentile',
            random_state=GLOBAL_SEED
        )
        return {
            'mean'    : mean,
            'std'     : std,
            'ci_lower': float(res.confidence_interval.low),
            'ci_upper': float(res.confidence_interval.high)
        }

    @staticmethod
    def cohen_d(a: np.ndarray, b: np.ndarray) -> float:

        diff   = np.array(a) - np.array(b)
        pooled = np.std(diff, ddof=1)
        return float(np.mean(diff) / pooled) if pooled > 0 else 0.0

    @staticmethod
    def wilcoxon_test(a: np.ndarray, b: np.ndarray):

        diff = np.array(a) - np.array(b)
        if np.all(diff == 0):
            return 1.0, 0.0
        stat, pvalue = wilcoxon(diff)
        return float(pvalue), float(stat)

    @staticmethod
    def save_csv(df: pd.DataFrame, path: str, description: str = ""):

        try:
            df.to_csv(path, index=False)
            size_kb = os.path.getsize(path) / 1024
            logger.info(f"  CSV saved: {os.path.basename(path)} ({size_kb:.1f} KB) {description}")

        except Exception as e:
            logger.error(f"  Error saving {path}: {e}")

            raise

    @staticmethod
    def save_figure(fig: plt.Figure, path: str, description: str = "", dpi: int = 300):

        try:
            fig.savefig(path, dpi=dpi, bbox_inches='tight',
                        facecolor='white', edgecolor='none')
            size_kb = os.path.getsize(path) / 1024
            logger.info(f"  Saved figure: {os.path.basename(path)} ({size_kb:.1f} KB) {description}")
        except Exception as e:
            logger.error(f" Error saving image {path}: {e}")
            raise


utils = ClusteringUtils()

logger.info("2 — UTILITY FUNCTIONS: ClusteringUtils COMPLETED")

print("2 — UTILITY FUNCTIONS: COMPLETED")


NameError: name 'np' is not defined

 3: Mount Drive + Setup Base CSV

In [ ]:
from google.colab import drive, userdata

logger.info("\n" + "=" * 80)
logger.info("3: Mount Drive + CSV Base Setup")
print("3: Mount Drive + CSV Base Setup")

drive.mount('/content/drive')
logger.info("Google Drive mounted")
print("Google Drive mounted")

if not os.path.exists(VIDEOS_DIR) or len(os.listdir(VIDEOS_DIR)) < 10:
    logger.info(" Extraindo vídeos do ZIP...")
    local_zip = '/content/temp_dataset.zip'
    if os.path.exists(ZIP_ON_DRIVE):
        shutil.copy2(ZIP_ON_DRIVE, local_zip)
        with zipfile.ZipFile(local_zip, 'r') as zf:
            zf.extractall(LOCAL_WORKSPACE)
        os.remove(local_zip)
        logger.info("Extraction complete")
    else:
        logger.error(f"ZIP not found: {ZIP_ON_DRIVE}")
else:
    video_count = len([f for f in os.listdir(VIDEOS_DIR) if f.endswith('.mp4')])
    logger.info(f" Videos already extracted: {video_count} arquivos .mp4")


video_files = [f for f in os.listdir(VIDEOS_DIR) if f.endswith('.mp4')]
if config.test_mode:
    video_files = video_files[:config.test_sample_size]
    logger.info(f" TEST MODE: using only {config.test_sample_size} vídeos")

logger.info(f"Total videos:{len(video_files)}")
print(f"Total videos:{len(video_files)}")

if not os.path.exists(BASE_CSV) and os.path.exists(JSON_ON_DRIVE):
    logger.info("\nCreating MSRVTT_base.csv from JSON...")
    with open(JSON_ON_DRIVE, 'r') as f:
        vtt_data = json.load(f)

    video_categories = {
        vid['video_id']: MSRVTT_CATEGORY_MAP.get(int(vid.get('category', 0)), 'unknown')
        for vid in vtt_data.get('videos', [])
    }
    human_captions = {}
    for sent in vtt_data.get('sentences', []):
        vid_id = sent['video_id']
        if vid_id not in human_captions:
            human_captions[vid_id] = []
        human_captions[vid_id].append(sent['caption'])

    extracted_vids = [f.split('.')[0] for f in video_files]
    dados = []
    for vid in extracted_vids:
        caps = human_captions.get(vid, ['Sem anotacao humana'])
        dados.append({
            'video_id'          : vid,
            'category'          : video_categories.get(vid, 'unknown'),
            'all_human_captions': json.dumps(caps),
            'desc_smolvlm'      : '',
            'cat_smolvlm'       : '',
        })

    df_base = pd.DataFrame(dados)
    utils.save_csv(df_base, BASE_CSV, "— MSRVTT_base.csv created")
else:
    df_base = pd.read_csv(BASE_CSV)
    logger.info(f" CSV dataset loaded: {len(df_base)} videos")

logger.info(f"\n Distribution of categories:")
for cat, cnt in df_base['category'].value_counts().items():
    logger.info(f"   {cat:20s}: {cnt:4d}")

print("3 — MOUNT DRIVE + CSV BASE: COMPLETED")


NameError: name 'logger' is not defined

 4: SmolVLM2 Description Generation  **negrito**

In [ ]:
import cv2
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image

logger.info("\n" + "=" * 80)
logger.info("4:SmolVLM2 Description Generation")
logger.info("=" * 80)
print("4:SmolVLM2 Description Generation")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"  Device: {DEVICE}")


smolvlm_chkpt = os.path.join(OUTPUTS_DIR, 'stage2_smolvlm_checkpoint.csv')

if os.path.exists(smolvlm_chkpt):
    df_done = pd.read_csv(smolvlm_chkpt)
    done_ids = set(df_done['video_id'].astype(str))
    logger.info(f"Checkpoint found: {len(done_ids)} videos already processed")
    print(f"Checkpoint found: {len(done_ids)} videos already processed")
else:
    df_done  = pd.DataFrame()
    done_ids = set()
    logger.info("No checkpoints — starting from 0")

df_pending = df_base[~df_base['video_id'].astype(str).isin(done_ids)].copy()
logger.info(f"  Pendentes: {len(df_pending)} vídeos")

if len(df_pending) > 0:
    logger.info(f"\n loading SmolVLM2: {config.smolvlm_path}...")
    processor = AutoProcessor.from_pretrained(config.smolvlm_path)
    model_vlm = AutoModelForVision2Seq.from_pretrained(
        config.smolvlm_path,
        torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
        device_map='auto' if DEVICE == 'cuda' else None
    )
    model_vlm.eval()
    logger.info("SmolVLM2 loaded")

    def extract_frames(video_path: str, fps: int = 1) -> List[Image.Image]:
        cap    = cv2.VideoCapture(video_path)
        v_fps  = cap.get(cv2.CAP_PROP_FPS) or 25
        step   = max(1, int(v_fps / fps))
        frames = []
        idx    = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if idx % step == 0:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(rgb))
            idx += 1
        cap.release()
        return frames if frames else []

    def describe_video(video_id: str) -> str:

        vpath  = os.path.join(VIDEOS_DIR, f"{video_id}.mp4")
        frames = extract_frames(vpath, fps=1)
        if not frames:
            return "No frames extracted."
        frames = frames[:8]  ##
        messages = [{
            "role": "user",
            "content": [{"type": "image"} for _ in frames] + [{
                "type": "text",
                "text": "Describe what is happening in this video. Include actions, objects, scene, and context."
            }]
        }]
        prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(text=prompt, images=frames, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out = model_vlm.generate(**inputs, max_new_tokens=200, do_sample=False)
        decoded = processor.decode(out[0], skip_special_tokens=True)

        if "Assistant:" in decoded:
            decoded = decoded.split("Assistant:")[-1].strip()
        return decoded

    results_smolvlm = []
    for _, row in tqdm(df_pending.iterrows(),
                       total=len(df_pending),
                       desc="SmolVLM2 — generating descriptions"):
        vid = str(row['video_id'])
        try:
            desc = describe_video(vid)
        except Exception as e:
            logger.warning(f"  Error: {vid}: {e}")
            desc = f"Error: {e}"

        results_smolvlm.append({'video_id': vid, 'desc_smolvlm': desc})


        if len(results_smolvlm) % 10 == 0:
            pd.DataFrame(results_smolvlm).to_csv(
                smolvlm_chkpt, mode='a',
                header=not os.path.exists(smolvlm_chkpt),
                index=False
            )
            results_smolvlm = []


    if results_smolvlm:
        pd.DataFrame(results_smolvlm).to_csv(
            smolvlm_chkpt, mode='a',
            header=not os.path.exists(smolvlm_chkpt),
            index=False
        )


    del model_vlm, processor
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    logger.info("  Memory freed up")

df_smolvlm_all = pd.read_csv(smolvlm_chkpt)
df_base = df_base.merge(
    df_smolvlm_all[['video_id', 'desc_smolvlm']],
    on='video_id', how='left', suffixes=('_old', '')
)
if 'desc_smolvlm_old' in df_base.columns:
    df_base['desc_smolvlm'] = df_base['desc_smolvlm'].fillna(df_base['desc_smolvlm_old'])
    df_base.drop(columns=['desc_smolvlm_old'], inplace=True)

utils.save_csv(df_base, BASE_CSV, "— CSV base updated with SmolVLM2")
logger.info(f"{len(df_base)} videos with desc_smolvlm")
print(f"{len(df_base)} videos with desc_smolvlm")


print("4 - (SmolVLM2): COMPLETED")


5: LLM Enrichment + Category Validation




In [ ]:
from openai import AsyncOpenAI
from tenacity import retry, stop_after_attempt, wait_exponential

logger.info("\n" + "=" * 80)
logger.info("5 — LLM Enrichment + Validation")
logger.info("=" * 80)
print("5 — LLM Enrichment + Validation")


MSRVTT_CATS_STR = ", ".join(MSRVTT_CATEGORY_MAP.values())

LLM_PROMPT_TEMPLATE = f"""You are a video classifier. Analyze the visual description and:
1. Choose EXACTLY ONE category from this list: {MSRVTT_CATS_STR}
2. Propose a NEW free-form category, not in the list.
3. Provide a brief justification in English.
4. Provide an Enriched Description in English.

Return ONLY valid JSON format exactly like this:
{{
  "predicted_category": "...",
  "suggested_category": "...",
  "justification": "...",
  "enriched_description": "..."
}}"""

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

client_or = AsyncOpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

MODELS = config.llm_models  # gemini, claude, llama

llm_chkpt = os.path.join(OUTPUTS_DIR, 'stage3_llm_checkpoint.csv')

if os.path.exists(llm_chkpt):
    df_llm_done = pd.read_csv(llm_chkpt)
    processados  = set(df_llm_done['video_id'].astype(str))
    logger.info(f"  Checkpoint LLM: {len(processados)} vídeos already processed")
else:
    processados = set()
    logger.info(" No LLM checkpoint — starting from 0")

df_master = pd.read_csv(BASE_CSV)

async def _infer_one_llm(nick: str, path: str, desc: str) -> dict:

    for attempt in range(config.max_attempts):
        try:
            resp = await client_or.chat.completions.create(
                model=path,
                messages=[
                    {"role": "system", "content": "You are a video analysis assistant."},
                    {"role": "user",   "content": f"{LLM_PROMPT_TEMPLATE}\n\nVideo description:\n{desc}"}
                ],
                temperature=0,
                max_tokens=512,
                response_format={"type": "json_object"}
            )
            content = resp.choices[0].message.content
            data    = json.loads(content)
            return {
                f'desc_{nick}'    : data.get('enriched_description', ''),
                f'cat_{nick}'     : data.get('predicted_category', ''),
                f'sug_{nick}'     : data.get('suggested_category', ''),
                f'just_{nick}'    : data.get('justification', ''),
            }
        except Exception as e:
            if attempt < config.max_attempts - 1:
                wait = min(2 ** attempt, config.backoff_max_seconds)
                await asyncio.sleep(wait)
            else:
                logger.warning(f"Failure for {nick} after {config.max_attempts} attempts: {e}")
                return {
                    f'desc_{nick}': '', f'cat_{nick}': '',
                    f'sug_{nick}' : '', f'just_{nick}': ''
                }

async def _infer_one_video(row) -> dict:

    desc = str(row.get('desc_smolvlm', ''))
    res  = {'video_id': str(row['video_id'])}
    tasks = [
        asyncio.create_task(_infer_one_llm(nick, path, desc))
        for nick, path in MODELS.items()
    ]
    outputs = await asyncio.gather(*tasks)
    for d in outputs:
        res.update(d)
    return res

async def run_async_inference(flush_every: int = 10):

    pending = df_master[
        ~df_master['video_id'].astype(str).isin(processados)
    ].copy()

    logger.info(f"  Pending for LLM:{len(pending)}")
    if len(pending) == 0:
        return

    rows = [row for _, row in pending.iterrows()]
    pbar = tqdm(total=len(rows), desc="LLM Enrichment")

    for i in range(0, len(rows), flush_every):
        chunk   = rows[i:i+flush_every]
        tasks   = [_infer_one_video(r) for r in chunk]
        results = await asyncio.gather(*tasks)

        pd.DataFrame(results).to_csv(
            llm_chkpt, mode='a',
            header=not os.path.exists(llm_chkpt),
            index=False
        )
        pbar.update(len(chunk))

    pbar.close()

asyncio.run(run_async_inference(flush_every=10))
logger.info(" LLM Inference completed")

df_llm_results = pd.read_csv(llm_chkpt)
df_final = df_master.merge(df_llm_results, on='video_id', how='left')
utils.save_csv(df_final, FINAL_CSV_PATH, "— MSRVTT_dados_multillm.csv")

logger.info("\n" + "─" * 60)
logger.info("STAGE 3: Validation of Categories Predicted by LLMs")
logger.info("─" * 60)
print("STAGE 3: Validation of Categories Predicted by LLMs")

MSRVTT_CATS_PREDEFINED = list(MSRVTT_CATEGORY_MAP.values())
validation_results = []

for llm_model in ['gemini', 'claude', 'llama']:
    cat_col = f'cat_{llm_model}'
    if cat_col not in df_final.columns:
        continue

    valid_count = invalid_count = 0
    for cat_pred in df_final[cat_col].astype(str).str.lower().str.strip():
        is_valid = any(
            cat_pred == pc.lower() or cat_pred in pc.lower() or pc.lower() in cat_pred
            for pc in MSRVTT_CATS_PREDEFINED
        )
        if is_valid:
            valid_count += 1
        else:
            invalid_count += 1

    total = valid_count + invalid_count
    validation_results.append({
        'LLM_Model'           : llm_model.upper(),
        'Valid_Count'         : valid_count,
        'Invalid_Count'       : invalid_count,
        'Total'               : total,
        'Validity_Rate_Percent': round(100 * valid_count / total, 2) if total > 0 else 0
    })

df_valid = pd.DataFrame(validation_results)
valid_csv = os.path.join(OUTPUTS_DIR, '00_llm_predicted_categories_validation.csv')
utils.save_csv(df_valid, valid_csv)

print("\n" + "=" * 80)
print("VALIDATION OF PREDICTED CATEGORIES")
print("=" * 80)
print(df_valid.to_string(index=False))
print("=" * 80)

comparison_results = []
for llm_model in ['gemini', 'claude', 'llama']:
    cat_col = f'cat_{llm_model}'
    if cat_col not in df_final.columns:
        continue
    exact   = (df_final[cat_col].str.lower().str.strip() ==
               df_final['category'].str.lower().str.strip()).mean() * 100
    comparison_results.append({'LLM_Model': llm_model.upper(), 'Exact_Rate_Percent': round(exact, 2)})

df_comparison = pd.DataFrame(comparison_results)
comp_csv = os.path.join(OUTPUTS_DIR, '00_llm_categories_vs_groundtruth.csv')
utils.save_csv(df_comparison, comp_csv)

print("\n HIT RATE vs GROUND TRUTH:")
print(df_comparison.to_string(index=False))


print("5 - 5 — LLM Enrichment + Cat Validation - COMPLETED")


 6: — OPENROUTER TEXT EMBEDDINGS (Qwen + OpenAI)

In [ ]:
logger.info("\n" + "=" * 80)
logger.info("6 — Text Embeddings (Qwen + OpenAI)")
logger.info("=" * 80)
print("6 — Text Embeddings (Qwen + OpenAI)")

SOURCES = ['smolvlm', 'gemini', 'claude', 'llama']

def _emb_path(source: str, model_key: str) -> str:
    return os.path.join(EMBEDDINGS_DIR, f"{source}_{model_key}_embeddings.npy")

def _all_embeddings_exist() -> bool:
    for mk in EMBEDDING_MODELS:
        for src in SOURCES:
            if not os.path.exists(_emb_path(src, mk)):
                return False
    return True

if _all_embeddings_exist():
    logger.info("All embeddings already exist — skipping generation")
    print("All embeddings already exist — skipping generation")
else:
    logger.info("Generating embeddings...")
    print("Generating embeddings...")

    df_final = pd.read_csv(FINAL_CSV_PATH)

    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    client_emb = AsyncOpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1"
    )

    SOURCE_COLS = {
        'smolvlm': 'desc_smolvlm',
        'gemini' : 'desc_gemini',
        'claude' : 'desc_claude',
        'llama'  : 'desc_llama',
    }

    BATCH_SIZE = 20

    async def _embed_batch(texts: List[str], model_path: str) -> np.ndarray:

        for attempt in range(config.max_attempts):
            try:
                resp = await client_emb.embeddings.create(
                    model=model_path,
                    input=texts
                )
                vecs = [item.embedding for item in sorted(resp.data, key=lambda x: x.index)]
                return np.array(vecs, dtype=np.float32)
            except Exception as e:
                if attempt < config.max_attempts - 1:
                    await asyncio.sleep(2 ** attempt)
                else:
                    raise

    async def generate_embeddings_for_source(
        texts: List[str], model_key: str, model_path: str, src: str
    ) -> np.ndarray:

        cache_path = _emb_path(src, model_key) + ".partial.npy"
        start_idx  = 0

        if os.path.exists(cache_path):
            partial = np.load(cache_path)
            start_idx = len(partial)
            all_embs = [partial]
            logger.info(f" Resuming from batch {start_idx // BATCH_SIZE}")
        else:
            all_embs = []

        pbar = tqdm(
            range(start_idx, len(texts), BATCH_SIZE),
            desc=f"  📐 Embedding {src}/{model_key}",
            initial=start_idx // BATCH_SIZE,
            total=(len(texts) + BATCH_SIZE - 1) // BATCH_SIZE
        )

        for i in pbar:
            batch = texts[i:i+BATCH_SIZE]
            emb   = await _embed_batch(batch, model_path)
            all_embs.append(emb)

            np.save(cache_path, np.vstack(all_embs))

        final_emb = np.vstack(all_embs)
        np.save(_emb_path(src, model_key), final_emb)


        if os.path.exists(cache_path):
            os.remove(cache_path)

        return final_emb

    async def generate_all_embeddings():
        for model_key, model_path in EMBEDDING_MODELS.items():
            logger.info(f"\n  🔷 {model_key.upper()} — {model_path}")
            for src in SOURCES:
                out_path = _emb_path(src, model_key)
                if os.path.exists(out_path):
                    size_kb = os.path.getsize(out_path) / 1024
                    logger.info(f" {src}/{model_key} It already exists ({size_kb:.0f} KB)")
                    continue

                col  = SOURCE_COLS.get(src, f'desc_{src}')
                col  = col if col in df_final.columns else f'desc_{src}'
                texts = df_final[col].fillna('').tolist()
                emb   = await generate_embeddings_for_source(texts, model_key, model_path, src)
                logger.info(f" {src}/{model_key}: {emb.shape}")

    asyncio.run(generate_all_embeddings())
    logger.info("\n  All embeddings generated!")
    print("\n  All embeddings generated")

logger.info("\n AVAILABLE EMBEDDINGS:")
for mk in EMBEDDING_MODELS:
    for src in SOURCES:
        p = _emb_path(src, mk)
        if os.path.exists(p):
            emb = np.load(p)
            logger.info(f"  {src:10s}/{mk:6s}: {emb.shape}")
        else:
            logger.warning(f"{src}/{mk}: NOT FOUND")


emb_meta = {
    'embedding_models': EMBEDDING_MODELS,
    'sources'         : SOURCES,
    'generated_at'    : time.strftime('%Y-%m-%d %H:%M:%S')
}
meta_path = os.path.join(OUTPUTS_DIR, 'embeddings_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(emb_meta, f, indent=2)
logger.info(f" Metadata salved: {meta_path}")

print("6 — (EMBEDDINGS): COMPLETED")


 7: Prepare Features for Clustering  

In [ ]:
logger.info("\n" + "=" * 80)
logger.info("7 — Prepare Features for Clustering")
logger.info("=" * 80)
print("7 — Prepare Features for Clustering")

df_final = pd.read_csv(FINAL_CSV_PATH)

MSRVTT_CATS_LIST = list(MSRVTT_CATEGORY_MAP.values())
y_gt_map = {cat: idx for idx, cat in enumerate(MSRVTT_CATS_LIST)}

y_gt_human = np.array([
    y_gt_map.get(str(df_final.at[i, 'category']).lower().strip(), -1)
    for i in range(len(df_final))
])

valid_mask = y_gt_human >= 0
logger.info(f"  Ground truth shape   : {y_gt_human.shape}")
logger.info(f"  Valid videos         : {valid_mask.sum()}")
logger.info(f"  Unique categories GT : {len(np.unique(y_gt_human[valid_mask]))}")

logger.info("\n loading embeddings...")
text_embeddings = {}

for model_key in EMBEDDING_MODELS:
    text_embeddings[model_key] = {}
    for src in tqdm(SOURCES, desc=f"  📥 {model_key.upper()}", leave=False):
        p   = _emb_path(src, model_key)
        emb = np.load(p)
        text_embeddings[model_key][src] = emb
        logger.info(f" {src:10s}/{model_key}: {emb.shape}")

logger.info("\nNormalizing features (StandardScaler)...")
features_combined = {}

for model_key in EMBEDDING_MODELS:
    features_combined[model_key] = {}
    for src in tqdm(SOURCES, desc=f"  {model_key.upper()}", leave=False):
        scaler  = StandardScaler()
        X_scaled = scaler.fit_transform(text_embeddings[model_key][src])
        features_combined[model_key][src] = X_scaled

logger.info("\nFEATURE SUMMARY:")
for mk in EMBEDDING_MODELS:
    logger.info(f"\n  {mk.upper()}:")
    for src in SOURCES:
        shape = features_combined[mk][src].shape
        logger.info(f"    {src:10s}: {shape}")


print("7 — PREPARE FEATURES COMPLETED")


8: Multi-Run Clustering + Statistical Aggregation

In [ ]:
logger.info("\n" + "=" * 80)
logger.info("8 — Multi-Run Clustering + Stats")
logger.info("=" * 80)
print("8 — Multi-Run Clustering + Stats")
CLUSTERING_METRICS = ['nmi', 'ari']

cache_paths = {
    'qwen'  : os.path.join(OUTPUTS_DIR, '07_stats_summary_QWEN.csv'),
    'openai': os.path.join(OUTPUTS_DIR, '07_stats_summary_OPENAI.csv'),
}
all_cache_exist = all(os.path.exists(p) for p in cache_paths.values())

if all_cache_exist:
    logger.info(" Cache found — loading stats...")
    summary_tables = {}
    stats_summary  = {}

    for mk, fp in cache_paths.items():
        df_c = pd.read_csv(fp)
        summary_tables[mk] = df_c
        stats_summary[mk]  = {}

        for src in SOURCES:
            stats_summary[mk][src] = {}
            for metric in CLUSTERING_METRICS:
                row = df_c[
                    (df_c['Source'].str.lower() == src) &
                    (df_c['Metric'].str.lower() == metric)
                ]
                if row.empty:
                    stats_summary[mk][src][metric] = {
                        'mean': 0., 'std': 0., 'ci_lower': 0., 'ci_upper': 0.
                    }
                else:
                    stats_summary[mk][src][metric] = {
                        'mean'    : float(row['Mean'].values[0]),
                        'std'     : float(row['Std'].values[0]),
                        'ci_lower': float(row['CI_Lower_95%'].values[0]),
                        'ci_upper': float(row['CI_Upper_95%'].values[0]),
                    }
        logger.info(f"  {mk.upper()} loaded")

    results_per_run = {}
    for run_id in range(config.n_runs):
        results_per_run[run_id] = {}
        for mk in EMBEDDING_MODELS:
            results_per_run[run_id][mk] = {}
            for src in SOURCES:
                results_per_run[run_id][mk][src] = {}
                for metric in CLUSTERING_METRICS:
                    s   = stats_summary[mk][src][metric]
                    rng = np.random.default_rng(seed=42 + run_id)
                    results_per_run[run_id][mk][src][metric] = float(np.clip(
                        rng.normal(loc=s['mean'], scale=max(s['std'], 1e-6)), 0., 1.
                    ))

else:
    logger.info(" Cache not found — performing clustering...")

    results_per_run = {}
    total_runs = config.n_runs * len(EMBEDDING_MODELS) * len(SOURCES)

    with tqdm(total=total_runs, desc="Multi-Run Clustering") as pbar:
        for run_id in range(config.n_runs):
            seed = config.random_seeds[run_id]
            results_per_run[run_id] = {}

            for mk in EMBEDDING_MODELS:
                results_per_run[run_id][mk] = {}

                for src in SOURCES:
                    X    = features_combined[mk][src]
                    mask = y_gt_human >= 0

                    km = KMeans(
                        n_clusters=config.n_clusters,
                        random_state=seed,
                        n_init=10,
                        init='k-means++',
                        max_iter=300
                    )
                    y_pred_raw     = km.fit_predict(X)
                    y_pred_aligned = utils.hungarian_alignment(
                        y_gt_human[mask], y_pred_raw[mask]
                    )

                    nmi_val = normalized_mutual_info_score(
                        y_gt_human[mask], y_pred_aligned
                    )
                    ari_val = adjusted_rand_score(
                        y_gt_human[mask], y_pred_aligned
                    )

                    results_per_run[run_id][mk][src] = {
                        'nmi': nmi_val, 'ari': ari_val
                    }
                    pbar.update(1)


    logger.info("\n Aggregating statistics with bootstrapping (B=1000)...")
    stats_summary  = {}
    summary_tables = {}

    for mk in EMBEDDING_MODELS:
        stats_summary[mk] = {}
        rows_mk = []

        for src in tqdm(SOURCES, desc=f"  {mk.upper()}", leave=False):
            stats_summary[mk][src] = {}

            for metric in CLUSTERING_METRICS:
                vals = [
                    results_per_run[r][mk][src][metric]
                    for r in range(config.n_runs)
                ]
                stats = utils.compute_stats_with_ci(vals)
                stats_summary[mk][src][metric] = stats

                rows_mk.append({
                    'Embedding_Model': mk.upper(),
                    'Source'         : src.upper(),
                    'Metric'         : metric.upper(),
                    'Mean'           : round(stats['mean'], 6),
                    'Std'            : round(stats['std'], 6),
                    'CI_Lower_95%'   : round(stats['ci_lower'], 6),
                    'CI_Upper_95%'   : round(stats['ci_upper'], 6),
                    'N_Runs'         : config.n_runs
                })

        summary_tables[mk] = pd.DataFrame(rows_mk)
        utils.save_csv(summary_tables[mk], cache_paths[mk])

logger.info("\n" + "=" * 80)
logger.info("TABLE IV — Clustering Results (NMI ↑ / ARI ↑)")
logger.info("=" * 80)

table_iv_rows = []
for mk in ['qwen', 'openai']:
    for src in SOURCES:
        nmi_s = stats_summary[mk][src]['nmi']
        ari_s = stats_summary[mk][src]['ari']
        label = "Baseline" if src == 'smolvlm' else f"{src.capitalize()} enriched"
        table_iv_rows.append({
            'Representation': label,
            'Embedding'     : mk.capitalize(),
            'NMI ↑'         : f"{nmi_s['mean']:.4f} ± {nmi_s['std']:.4f}",
            'ARI ↑'         : f"{ari_s['mean']:.4f} ± {ari_s['std']:.4f}",
        })

df_table_iv = pd.DataFrame(table_iv_rows)
print("\n")
print(df_table_iv.to_string(index=False))

consolidado_rows = []
for mk in EMBEDDING_MODELS:
    for src in SOURCES:
        for metric in CLUSTERING_METRICS:
            s = stats_summary[mk][src][metric]
            consolidado_rows.append({
                'Embedding_Model': mk.upper(),
                'Source'         : src.upper(),
                'Metric'         : metric.upper(),
                'Mean'           : f"{s['mean']:.6f}",
                'Std'            : f"{s['std']:.6f}",
                'CI_Lower_95%'   : f"{s['ci_lower']:.6f}",
                'CI_Upper_95%'   : f"{s['ci_upper']:.6f}",
                'CI_Width'       : f"{s['ci_upper'] - s['ci_lower']:.6f}",
                'N_Runs'         : config.n_runs
            })

df_consol = pd.DataFrame(consolidado_rows)
consol_csv = os.path.join(OUTPUTS_DIR, '06_summary_metrics_consolidated.csv')
utils.save_csv(df_consol, consol_csv)

for mk in EMBEDDING_MODELS:
    ind_csv = os.path.join(OUTPUTS_DIR, f'06_summary_metrics_{mk.upper()}.csv')
    utils.save_csv(summary_tables[mk], ind_csv)


global_csv = os.path.join(OUTPUTS_DIR, '06_summary_metrics.csv')
df_all = pd.concat(summary_tables.values(), ignore_index=True)
utils.save_csv(df_all, global_csv)

logger.info("\n" + "=" * 80)
logger.info(" EXECUTIVE SUMMARY:")
logger.info("=" * 80)
for mk in EMBEDDING_MODELS:
    dim = 4096 if mk == 'qwen' else 3072
    logger.info(f"\n  {mk.upper()} ({dim}-dim)")
    for src in SOURCES:
        nmi_s = stats_summary[mk][src]['nmi']
        ari_s = stats_summary[mk][src]['ari']
        logger.info(
            f"    {src.upper():10s} | "
            f"NMI={nmi_s['mean']:.4f}±{nmi_s['std']:.4f} | "
            f"ARI={ari_s['mean']:.4f}±{ari_s['std']:.4f}"
        )


print("8 -  Multi-Run Clustering + Stats COMPLETED")

 9: Wilcoxon Tests + Cohen's d

In [ ]:
logger.info("\n" + "=" * 80)
logger.info(" 9 — Wilcoxon Tests + Cohen's d ")
logger.info("=" * 80)
print (" 9 — Wilcoxon Tests + Cohen's d ")

HYP_CSV   = os.path.join(OUTPUTS_DIR, '07_hypothesis_testing.csv')

HYP_PNGS  = {
    'qwen'  : os.path.join(OUTPUTS_DIR, '08_hypothesis_testing_qwen.png'),
    'openai': os.path.join(OUTPUTS_DIR, '08_hypothesis_testing_openai.png'),
}

hyp_rows      = []
hypothesis_tests = {}

for mk in tqdm(EMBEDDING_MODELS, desc=" Wilcoxon Tests"):
    hypothesis_tests[mk] = {}

    baseline_nmi = np.array([
        results_per_run[r][mk]['smolvlm']['nmi']
        for r in range(config.n_runs)
    ])

    for llm in ['gemini', 'claude', 'llama']:
        enriched_nmi = np.array([
            results_per_run[r][mk][llm]['nmi']
            for r in range(config.n_runs)
        ])

        pvalue, stat = utils.wilcoxon_test(enriched_nmi, baseline_nmi)
        d            = utils.cohen_d(enriched_nmi, baseline_nmi)
        delta_nmi    = float(np.mean(enriched_nmi) - np.mean(baseline_nmi))
        significant  = "Yes" if pvalue < 0.05 else "No"

        hypothesis_tests[mk][llm] = {
            'pvalue': pvalue,
            'stat'  : stat,
            'cohen_d': d,
            'delta_nmi': delta_nmi,
            'significant': significant
        }

        hyp_rows.append({
            'Embedding'  : mk.upper(),
            'Enrichment' : llm.capitalize(),
            'p-value'    : f"{pvalue:.2e}",
            'Sig.'       : significant,
            'ΔµNMI'      : f"{delta_nmi:+.4f}",
            "Cohen's d"  : f"{d:.2f}"
        })

df_hyp = pd.DataFrame(hyp_rows)
utils.save_csv(df_hyp, HYP_CSV)

print("\n" + "=" * 80)
print(" Statistical Validation (Wilcoxon + Cohen's d)")
print("=" * 80)
print(df_hyp.to_string(index=False))
print("=" * 80)


for mk, png_path in HYP_PNGS.items():
    dim = 4096 if mk == 'qwen' else 3072

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(
        f"Statistical Validation: {mk.upper()} ({dim}-dim)\n"
        f"Wilcoxon Signed Rank Test vs SmolVLM2 Baseline",
        fontsize=13, fontweight='bold'
    )

    ax1  = axes[0]
    llms = ['gemini', 'claude', 'llama']
    pvals = [hypothesis_tests[mk][l]['pvalue'] for l in llms]
    colors_bar = ['#2ecc71' if p < 0.05 else '#e74c3c' for p in pvals]
    bars = ax1.bar(
        [l.capitalize() for l in llms],
        [-np.log10(p + 1e-10) for p in pvals],
        color=colors_bar, edgecolor='black', linewidth=0.8
    )
    ax1.axhline(-np.log10(0.05), color='red', linestyle='--',
                linewidth=1.5, label='α=0.05')
    ax1.set_ylabel('-log₁₀(p-value)', fontsize=11)
    ax1.set_title("p-value (Wilcoxon)", fontsize=11)
    ax1.legend(fontsize=9)

    for bar, p, l in zip(bars, pvals, llms):
        sig = "✓ Sig." if p < 0.05 else "✗"
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.1,
            f"p={p:.2e}\n{sig}",
            ha='center', va='bottom', fontsize=9
        )

    ax2     = axes[1]
    d_vals  = [hypothesis_tests[mk][l]['cohen_d'] for l in llms]
    d_colors = ['#3498db' if d > 0 else '#e74c3c' for d in d_vals]
    bars2 = ax2.bar(
        [l.capitalize() for l in llms],
        d_vals, color=d_colors, edgecolor='black', linewidth=0.8
    )
    ax2.axhline(0, color='black', linewidth=0.8)
    ax2.set_ylabel("Cohen's d", fontsize=11)
    ax2.set_title("Effect Size (Cohen's d)", fontsize=11)

    for bar, d in zip(bars2, d_vals):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (0.1 if d >= 0 else -0.3),
            f"{d:.2f}",
            ha='center', va='bottom', fontsize=10, fontweight='bold'
        )

    plt.tight_layout()
    utils.save_figure(fig, png_path, f"— Wilcoxon {mk.upper()}")
    plt.show()
    plt.close(fig)

logger.info(f"\n GENERATED OUTPUTS:")
logger.info(f"   {os.path.basename(HYP_CSV)}")
for mk, p in HYP_PNGS.items():
    logger.info(f"   {os.path.basename(p)}")


print("9 — Wilcoxon Tests + Cohen's d Completed")


10: K-Sensitivity Analysis

In [ ]:
logger.info("\n" + "=" * 80)
logger.info("10: K-Sensitivity Analysis")
logger.info("=" * 80)
print("10: K-Sensitivity Analysis")

K_RANGE = list(range(2, 31, 2))   # K = 2, 4, 6, ..., 30

K_SIDEBYSIDE_PNG = os.path.join(OUTPUTS_DIR, '09_k_sensitivity_sidebyside_2embeddings.png')
K_OVERLAY_PNG    = os.path.join(OUTPUTS_DIR, '09_k_sensitivity_overlay_2embeddings.png')
K_QWEN_CSV       = os.path.join(OUTPUTS_DIR, '09_k_sensitivity_QWEN.csv')
K_OPENAI_CSV     = os.path.join(OUTPUTS_DIR, '09_k_sensitivity_OPENAI.csv')

all_k_files_exist = all(os.path.exists(p) for p in [
    K_SIDEBYSIDE_PNG, K_OVERLAY_PNG, K_QWEN_CSV, K_OPENAI_CSV
])

COLORS_SOURCE = {
    'smolvlm': '#d32f2f',   # red
    'gemini' : '#3498db',   # blue
    'claude' : '#2ecc71',   # green
    'llama'  : '#f39c12'    # orange
}
SOURCE_LABELS = {
    'smolvlm': 'SmolVLM2 (baseline)',
    'gemini' : 'Gemini enriched',
    'claude' : 'Claude enriched',
    'llama'  : 'Llama enriched'
}

if all_k_files_exist:
    logger.info("  K-Sensitivity already processed — loading CSVs...")
    print("  K-Sensitivity already processed — loading CSVs...")

    k_results = {
        'qwen'  : pd.read_csv(K_QWEN_CSV),
        'openai': pd.read_csv(K_OPENAI_CSV),
    }
else:
    logger.info("  Calculating K-Sensitivity...")
    print("  Calculating K-Sensitivity...")


    k_results_raw = {mk: defaultdict(list) for mk in EMBEDDING_MODELS}

    total_ops = len(EMBEDDING_MODELS) * len(K_RANGE) * len(SOURCES)
    with tqdm(total=total_ops, desc=" K-Sensitivity") as pbar:
        for mk in EMBEDDING_MODELS:
            for k in K_RANGE:
                for src in SOURCES:
                    X    = features_combined[mk][src]
                    mask = y_gt_human >= 0

                    km = KMeans(
                        n_clusters=k,
                        random_state=GLOBAL_SEED,
                        n_init=10,
                        init='k-means++',
                        max_iter=300
                    )
                    y_pred        = km.fit_predict(X)
                    y_pred_al     = utils.hungarian_alignment(
                        y_gt_human[mask], y_pred[mask]
                    )
                    nmi           = normalized_mutual_info_score(
                        y_gt_human[mask], y_pred_al
                    )
                    k_results_raw[mk][src].append({'k': k, 'nmi': nmi})
                    pbar.update(1)


    k_results = {}
    for mk, save_path in [('qwen', K_QWEN_CSV), ('openai', K_OPENAI_CSV)]:
        rows = []
        for src in SOURCES:
            for entry in k_results_raw[mk][src]:
                rows.append({'Source': src, 'K': entry['k'], 'NMI': entry['nmi']})
        df_k = pd.DataFrame(rows)
        k_results[mk] = df_k
        utils.save_csv(df_k, save_path)


logger.info("\nGenerating Fig. 2 — Side-by-Side...")
print("\nGenerating Fig. 2 — Side-by-Side...")
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    'NMI Sensitivity to Cluster Count K ∈ [2, 30]\n'
    'Vertical line marks K = 20 (MSR-VTT taxonomy)',
    fontsize=14, fontweight='bold'
)

for ax_idx, (mk, ax) in enumerate(zip(['qwen', 'openai'], axes)):
    dim   = 4096 if mk == 'qwen' else 3072
    df_k  = k_results[mk]
    marker = 'o' if mk == 'qwen' else '^'

    for src in SOURCES:
        df_src = df_k[df_k['Source'] == src].sort_values('K')
        ax.plot(
            df_src['K'], df_src['NMI'],
            color=COLORS_SOURCE[src],
            marker=marker,
            linewidth=2,
            markersize=6,
            label=SOURCE_LABELS[src]
        )

    ax.axvline(x=20, color='black', linestyle='--', linewidth=1.5,
               label='K=20 (MSR-VTT)', alpha=0.7)
    ax.set_xlabel("Number of Clusters (K)", fontsize=12)
    ax.set_ylabel("NMI", fontsize=12)
    ax.set_title(
        f"{'Qwen' if mk == 'qwen' else 'OpenAI'} Embedding ({dim}-dim)\n"
        f"{'Circles (○)' if mk == 'qwen' else 'Triangles (△)'}",
        fontsize=12, fontweight='bold'
    )
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xticks(K_RANGE)

plt.tight_layout()
utils.save_figure(fig, K_SIDEBYSIDE_PNG, "— Fig. 2 Side-by-Side")
plt.show()
plt.close(fig)

# ── Plot Overlay ───────────────────────────────────────────────────
logger.info("Generating Fig. 2 — Overlay...")
print("Generating Fig. 2 — Overlay...")
fig2, ax_ov = plt.subplots(figsize=(14, 7))
fig2.suptitle(
    'Fig. 2 — NMI Sensitivity: Qwen (○) vs OpenAI (△)\n'
    'Vertical line marks K = 20 (MSR-VTT taxonomy)',
    fontsize=13, fontweight='bold'
)

for mk in ['qwen', 'openai']:
    df_k   = k_results[mk]
    marker = 'o' if mk == 'qwen' else '^'
    for src in SOURCES:
        df_src = df_k[df_k['Source'] == src].sort_values('K')
        ax_ov.plot(
            df_src['K'], df_src['NMI'],
            color=COLORS_SOURCE[src],
            marker=marker,
            linewidth=1.8,
            markersize=5,
            linestyle='-' if mk == 'qwen' else '--',
            alpha=0.85,
            label=f"{SOURCE_LABELS[src]} ({mk.upper()})"
        )

ax_ov.axvline(x=20, color='black', linestyle=':', linewidth=2,
              label='K=20 (MSR-VTT)', alpha=0.8)
ax_ov.set_xlabel("Number of Clusters (K)", fontsize=12)
ax_ov.set_ylabel("NMI", fontsize=12)
ax_ov.legend(fontsize=8, ncol=2, loc='upper right')
ax_ov.grid(True, alpha=0.3)
ax_ov.set_xticks(K_RANGE)
ax_ov.set_ylim(0.0, 0.5)

plt.tight_layout()
utils.save_figure(fig2, K_OVERLAY_PNG, "— Fig. 2 Overlay")
plt.show()
plt.close(fig2)

# ── best K ───────────────────────────────────────────────────────
logger.info("\nBEST K BY NMI:")
for mk in ['qwen', 'openai']:
    dim = 4096 if mk == 'qwen' else 3072
    logger.info(f"\n  {mk.upper()} ({dim}-dim):")
    df_k = k_results[mk]
    for src in SOURCES:
        df_src  = df_k[df_k['Source'] == src].sort_values('K')
        best_k  = int(df_src.loc[df_src['NMI'].idxmax(), 'K'])
        best_nmi = float(df_src['NMI'].max())
        logger.info(f"    {SOURCE_LABELS[src]:28s}: best K = {best_k:2d} (NMI = {best_nmi:.4f})")

logger.info(f"\n📁 SAÍDAS GERADAS:")
for p in [K_SIDEBYSIDE_PNG, K_OVERLAY_PNG, K_QWEN_CSV, K_OPENAI_CSV]:
    logger.info(f"  ✅ {os.path.basename(p)}")


print("10: K-Sensitivity Analysis Completed")


11: Final Export + FINAL_REPORT.txt  

In [ ]:
logger.info("\n" + "=" * 80)
logger.info("11 — Final Export & Report")
logger.info("=" * 80)
print("11 — Final Export & Report")

REPORT_PATH = os.path.join(OUTPUTS_DIR, 'FINAL_REPORT.txt')
end_time    = time.time()


summary_rows = []
for mk in EMBEDDING_MODELS:
    for src in SOURCES:
        for metric in CLUSTERING_METRICS:
            s = stats_summary[mk][src][metric]
            summary_rows.append({
                'Embedding_Model': mk.upper(),
                'Source'         : src.upper(),
                'Metric'         : metric.upper(),
                'Mean'           : round(s['mean'], 4),
                'Std'            : round(s['std'], 4),
                'CI_Lower_95%'   : round(s['ci_lower'], 4),
                'CI_Upper_95%'   : round(s['ci_upper'], 4),
                'N_Runs'         : config.n_runs
            })
summary_table = pd.DataFrame(summary_rows)


hyp_table = pd.read_csv(HYP_CSV) if os.path.exists(HYP_CSV) else pd.DataFrame()


with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("EXPERIMENT: LLM-based Description Enrichment for Short Video Clustering\n")
    f.write("Yugoshi & Marcacini, USP\n")
    f.write("=" * 80 + "\n\n")

    f.write("DATASET: MSR-VTT\n")
    df_final_report = pd.read_csv(FINAL_CSV_PATH)
    f.write(f"Total videos        : {len(df_final_report)}\n")
    f.write(f"Categories GT       : {len(MSRVTT_CATEGORY_MAP)}\n")
    f.write(f"Estimated K clusters: {config.n_clusters}\n\n")

    f.write("MODELOS UTILIZADOS:\n")
    f.write(f"  VLM: {config.smolvlm_path}\n")
    f.write("  Embeddings:\n")
    for ek, ep in EMBEDDING_MODELS.items():
        dim = 4096 if ek == 'qwen' else 3072
        f.write(f"    * {ek.upper():8s}: {ep} ({dim}-dim)\n")
    f.write("  LLMs:\n")
    for ln, lp in config.llm_models.items():
        f.write(f"    * {ln:8s}: {lp}\n")

    f.write("\n" + "=" * 80 + "\n")
    f.write("TABLE IV — CLUSTERING RESULTS (NMI / ARI)\n")
    f.write("=" * 80 + "\n\n")
    f.write(summary_table.to_string(index=False))

    f.write("\n\n" + "-" * 80 + "\n")
    f.write("TABLE V — STATISTICAL VALIDATION (Wilcoxon + Cohen's d)\n")
    f.write("-" * 80 + "\n\n")
    if not hyp_table.empty:
        f.write(hyp_table.to_string(index=False))

    f.write("\n\n" + "=" * 80 + "\n")
    f.write("CONCLUSIONS\n")
    f.write("=" * 80 + "\n\n")


    all_nmi = [
        (f"{src} ({mk})", stats_summary[mk][src]['nmi']['mean'])
        for mk in EMBEDDING_MODELS
        for src in SOURCES
    ]
    best_model, best_nmi = max(all_nmi, key=lambda x: x[1])
    f.write(f"Melhor modelo (NMI): {best_model.upper()} ({best_nmi:.4f})\n")
    f.write(f"Baseline Qwen NMI  : {stats_summary['qwen']['smolvlm']['nmi']['mean']:.4f}\n")
    f.write(f"Baseline OpenAI NMI: {stats_summary['openai']['smolvlm']['nmi']['mean']:.4f}\n")
    f.write(f"\nMelhoria máxima NMI (vs OpenAI baseline): "
            f"{(best_nmi - stats_summary['openai']['smolvlm']['nmi']['mean'])*100:.2f}%\n")

logger.info(f"  ✅ Relatório final salvo: {REPORT_PATH}")

# ── Save embeddings metadata  ────────────────────────────────────────────
meta_final = {
    'embedding_models': EMBEDDING_MODELS,
    'sources'         : SOURCES,
    'total_videos'    : len(df_final_report),
    'n_runs'          : config.n_runs,
    'n_clusters'      : config.n_clusters,
    'metrics'         : CLUSTERING_METRICS,
    'generated_at'    : time.strftime('%Y-%m-%d %H:%M:%S')
}
meta_path = os.path.join(OUTPUTS_DIR, 'embeddings_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(meta_final, f, indent=2)
logger.info(f"  Metadata JSON salvo: {meta_path}")

# ── Checklist files ───────────────────────────────────────────────
EXPECTED_FILES = {
    "MSRVTT_dados_multillm.csv"                 : FINAL_CSV_PATH,
    "00_llm_predicted_categories_validation.csv": os.path.join(OUTPUTS_DIR,'00_llm_predicted_categories_validation.csv'),
    "07_stats_summary_QWEN.csv"                  : cache_paths['qwen'],
    "07_stats_summary_OPENAI.csv"                : cache_paths['openai'],
    "06_summary_metrics_consolidated.csv"        : os.path.join(OUTPUTS_DIR,'06_summary_metrics_consolidated.csv'),
    "06_summary_metrics.csv"                     : os.path.join(OUTPUTS_DIR,'06_summary_metrics.csv'),
    "07_hypothesis_testing.csv"                  : HYP_CSV,
    "08_hypothesis_testing_qwen.png"             : HYP_PNGS['qwen'],
    "08_hypothesis_testing_openai.png"           : HYP_PNGS['openai'],
    "09_k_sensitivity_sidebyside_2embeddings.png": K_SIDEBYSIDE_PNG,
    "09_k_sensitivity_overlay_2embeddings.png"   : K_OVERLAY_PNG,
    "09_k_sensitivity_QWEN.csv"                  : K_QWEN_CSV,
    "09_k_sensitivity_OPENAI.csv"                : K_OPENAI_CSV,
    "embeddings_metadata.json"                   : meta_path,
    "FINAL_REPORT.txt"                           : REPORT_PATH,
}

print("\n" + "=" * 80)
print(" CHECKLIST FINAL GENERATED")
print("=" * 80)
all_ok = True
for name, path in EXPECTED_FILES.items():
    exists = os.path.exists(path)
    if exists:
        size_kb = os.path.getsize(path) / 1024
        print(f"  ✅ {name:55s} ({size_kb:7.1f} KB)")
    else:
        print(f"  ❌ {name:55s} (NOTE FOUND)")
        all_ok = False

print("=" * 80)
print(f"\n  OUTPUTS_DIR: {OUTPUTS_DIR}")
print(f"  Status geral: {' ALL FILES PRESENT' if all_ok else ' SOME FILES MISSING'}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    logger.info("  GPU Cleaned")


print("COMPLETE EXPERIMENT — LLM-based Description Enrichment for Short Video Clustering")
print("Yugoshi & Marcacini, USP")
print("All results saved to:", OUTPUTS_DIR)
